# **ColabSeqDisplay: fit a model to your own variant library, in a browser.**

<img src="https://img.shields.io/badge/Paper-not%20yet%20posted-lightgrey" style="max-width: 100%;">
<a href="https://github.com/JasonJiangs/ColabSeqDisplay"><img src="https://img.shields.io/badge/Github-black?logo=github" style="max-width: 100%;"></a>
<a href="https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay.ipynb"><img src="https://img.shields.io/badge/Open%20in-Colab-F9AB00?logo=googlecolab&logoColor=white" style="max-width: 100%;"></a>
<a href="https://github.com/JasonJiangs/ColabSeqDisplay"><img src="https://img.shields.io/badge/License-see%20repository-lightgrey" style="max-width: 100%;"></a>

- You measured a **combinatorial variant library** — one row per variant, one column per mutated site, one column per assay condition. This notebook fine-tunes a protein language model on it with LoRA, checks the result against a one-hot floor that any linear model could reach, and hands you something that scores variants you have not made yet.

- **It is for the person who ran the assay.** No code, no YAML, no conda: every choice is a field in the panel below, and the hyperparameters for each (backbone, pooling) pair are read from a registry rather than tuned by you.

- **A trained model is a file.** It leaves here as `model_bundle.zip` — LoRA weights, the head, your library spec, the frozen pooling positions and the provenance, a few megabytes. Download it, keep it, email it. There is no hub, no account and no upload: nothing you load leaves this runtime.

- **Three notebooks.** [ColabSeqDisplay](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay.ipynb) trains, evaluates and exports a model. [ColabSeqDisplay_Prepare](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay_Prepare.ipynb) is run once per new protein and makes the two optional input files. [ColabSeqDisplay_Predict](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay_Predict.ipynb) scores new variants with a model you already trained.

- **The science is not ours.** LoRA injection, the training loop, pooling, metrics and splits come from the *SequenceDisplay-Workflow-Optimization* research package (`seqdisplay_opt`), vendored into `colabsd/engine/` so that this notebook installs one repository and runs. Every module there names the file it came from; `ATTRIBUTION.md` collects them.

<font color="red">⚠️ <b>Before you run anything:</b> the run-button cell installs this package from https://github.com/JasonJiangs/ColabSeqDisplay.git. If your runtime cannot reach that address, the cell stops there and quotes what `git` or `pip` said — put a fork's URL, or the path of a folder you uploaded to this runtime, in the field at the top of that cell.</font>

# How to start

## 1 · Switch this runtime to a GPU

`Runtime` ▸ `Change runtime type` ▸ **T4 GPU** ▸ `Save`. Colab restarts the runtime, which takes a few seconds and clears anything you had already run.

## 2 · Click the run-button

Hover over the cell below and click ▶ on its left. The cell installs the package (1–2 minutes the first time) and then draws the panel that **is** this notebook: every choice you make is a field or a button in it. There is no code to write and no other cell to edit.

**This notebook has two buttons, not one.** The panel reports validation numbers only. Reading the locked test partition is the second cell, at the bottom — a separate, counted, deliberate act. That cell explains why.

## 3 · Which GPU

- **T4 — <font color="red">free</font>.** 16 GB, which is enough for most of this. <font color="red">It is also unstable: free sessions are pre-empted, drop their connection, and are capped in length, so a run measured in hours is a run you will probably lose halfway.</font> Mount Drive first — see below — and a dropped session costs you the run in progress rather than everything you have done.
- **L4 (needs Colab Pro).** 24 GB — the smallest card that runs the three backbones the registry says will not fit a T4 (`SaProt-1.3B`, `ProtT5-XL`, `Ankh-large`) at all, and enough session stability to finish a long run. Slower than an A100.
- **A100 (needs Colab Pro).** 40 GB and much faster. This is the card for a full multi-seed evaluation of a large backbone.
- **No GPU (CPU runtime).** The panel still opens, still loads and checks your library, and still runs the one-hot floor and the report. Anything that has to run the language model itself refuses to start and says which runtime it needs, rather than dying halfway through a training loop.

### The free tier will disconnect. Mount Drive before it does.

`Files` — the folder icon in the left margin — then **Mount Drive**, and let Colab run the cell it offers you. Mounting does not stop the disconnect. It gives you one folder, `/content/drive/MyDrive/`, that survives one: everything else in the runtime is thrown away when the session ends, including `/content`, the multi-gigabyte backbone download and anything the panel had written. So mount it before you start something long, and copy what you want to keep into it as it appears — `model_bundle.zip` above all, which is small and is the whole trained model.

### Which backbone fits which card

| backbone | pooled feature | needs a 3Di string | est. min per run on a T4 | where to run it |
|---|---|---|---|---|
| `ESM2-8M` | 320-d | no | 20 | a free T4 |
| `ESM2-35M` | 480-d | no | 40 | a free T4 |
| `ESM2-150M` | 640-d | no | 90 | a free T4, if you mount Drive first |
| `ESM2-650M` | 1280-d | no | 240 | **L4 or A100** — a free session usually drops first |
| `SaProt-35M` | 480-d | **yes** | 45 | a free T4 |
| `SaProt-650M` | 1280-d | **yes** | 260 | **L4 or A100** — a free session usually drops first |
| `SaProt-1.3B` | 1280-d | **yes** | — | **L4 or A100** — will not fit a T4 |
| `ProtT5-XL` | 1024-d | no | — | **L4 or A100** — will not fit a T4 |
| `Ankh-large` | 1536-d | no | — | **L4 or A100** — will not fit a T4 |
| `ESMC-300M` | 960-d | no | 120 | a free T4, if you mount Drive first |
| `ESMC-600M` | 1152-d | no | 220 | **L4 or A100** — a free session usually drops first |
| `SeqDance` | 480-d | no | 45 | a free T4 |
| `ESMDance` | 50-d | no | 45 | a free T4 |

**The minutes are estimates, not measurements.** They come from the backbone registry, which quotes an order-of-magnitude wall-clock figure for **one** training run (one data split, one model seed) over the bundled SlugCas9 example — 16,424 variants of a 1,054-residue protein — on a Colab T4. Twice the rows costs roughly twice the time; a longer protein costs more than proportionally, because attention does. Nobody has timed your card, and no run in this project has ever been timed inside a real Colab runtime; the panel prints the real rate once it is training, and that is the number to trust. The registry's own evaluation protocol is 3 x 3 = 9 runs, so multiply the column by 9 before you plan a full one.

**The last column is this repository's reading of those estimates**, not a measurement either: ≤ 60 min per run is comfortable on a free T4; up to 150 min is fine if your results are on Drive; beyond that a free session is likely to end before the run does. A blank estimate means the registry says the model does not fit a T4's 16 GB at all.

**Hyperparameters are looked up, not tuned by you.** 10 of the 28 (backbone, pooling) pairs in the registry carry tuned values, all of them for `cosine_p90_mean` pooling, scoring 0.5478–0.5636 test Spearman on upstream's benchmark. 9 of those 10 are backbones you can run here:

- `ESM2-150M` · `cosine_p90_mean` — 0.5546
- `ESM2-35M` · `cosine_p90_mean` — 0.5486
- `ESM2-650M` · `cosine_p90_mean` — 0.5636
- `ESMC-300M` · `cosine_p90_mean` — 0.5487
- `ESMDance` · `cosine_p90_mean` — 0.5478
- `ProtT5-XL` · `cosine_p90_mean` — 0.5583
- `SaProt-1.3B` · `cosine_p90_mean` — 0.5578
- `SaProt-35M` · `cosine_p90_mean` — 0.5576
- `SaProt-650M` · `cosine_p90_mean` — 0.5592

Every other pair is a **placeholder**: the panel raises a **Warning** the moment you pick one, and every number it prints afterwards is a lower bound rather than a result. Each score above is an average over 3 data splits x 3 model seeds = 9 training runs; the panel defaults to **one** run, which is enough to see whether the machinery works and not enough to quote a ±.

**Structure.** The backbones marked *needs a 3Di string* read structure as well as sequence, so they need one extra input file for your wild type. [ColabSeqDisplay_Prepare](https://colab.research.google.com/github/JasonJiangs/ColabSeqDisplay/blob/main/colab/ColabSeqDisplay_Prepare.ipynb) makes it. Every other backbone needs nothing but the sequence.

**Extra installs.** Three of these backbones need a package Colab does not ship. The registry says which, in its own words:

- `ProtT5-XL` — Encoder only (~1.2B parameters); needs an L4/A100. The weights load through plain transformers, but the slow sentencepiece tokenizer is mandatory for this repo: pip install sentencepiece.
- `ESMC-300M` — Needs the EvolutionaryScale SDK: pip install esm.
- `ESMC-600M` — Needs the EvolutionaryScale SDK: pip install esm.

**Not offered here.** `METL` is in the registry but not in the panel: Rosetta-pretrained and protein-specific: no HuggingFace weights, so it is out of scope for Colab.

In [ ]:
#@title **Click the run-button to use ColabSeqDisplay** { display-mode: "form" }

#@markdown ### Hint
#@markdown - The first run of this cell installs ColabSeqDisplay: **1-2 minutes** on a fresh runtime, with nothing for you to do while it works. Run it again later in the same session and it skips straight to the panel.
#@markdown - **The panel this cell draws below itself is the whole program.** Answer what it asks, press the buttons it offers. There is no code to write and nothing else in this notebook to edit.
#@markdown - **What the run-button is telling you.** The ▶ arrow means nothing is running: click it to start. It spins while the cell installs the package and builds the panel, then goes back to ▶ — that means finished, not broken. The panel stays live after the cell ends, for as long as this runtime does; if it ever stops responding, click ▶ again to rebuild it.
#@markdown ### <font color=red>If the session disconnects</font>
#@markdown - <font color=red>Colab drops long sessions, and the free T4 drops them soonest. Reconnect, run this cell again, and the panel comes back — but the runtime is empty: whatever was under `/content` is gone, whatever you wrote to a mounted Google Drive folder is not. Mount Drive before you start anything long.</font>
#@markdown - <font color=red>Changing the runtime type restarts Python and empties it just the same. Stop this cell first, change the runtime, then run it again.</font>
#@markdown ### Where the code comes from
#@markdown - The field below names what gets installed — **one repository, and it is the whole program**: the fine-tuning engine ships inside it. A **folder path** works as well as a URL — the path of a checkout you uploaded to this runtime — and is installed with `pip install -e`, which keeps its `config/best/` registry and its bundled `examples/` where you can read and edit them.
#@markdown - <font color=red>If this runtime cannot reach that address the cell stops there, quotes what `git` or `pip` said, and names this field as the one to change.</font>
colabsd_repository = "https://github.com/JasonJiangs/ColabSeqDisplay.git"  #@param {type:"string"}

import importlib
import importlib.util
import subprocess
import sys
from pathlib import Path

WORK_ROOT = Path.cwd()


def run_command(command):
    """Run a command, raising with its own output when it fails."""
    parts = [str(part) for part in command]
    finished = subprocess.run(parts, capture_output=True, text=True)
    if finished.returncode != 0:
        raise RuntimeError(
            "This command failed:\n  " + " ".join(parts) + "\n"
            + (finished.stdout or "")[-1500:] + (finished.stderr or "")[-1500:]
        )
    return finished


def checkout(source, name):
    """A local folder as given, or a shallow clone of a git URL beside this notebook."""
    local = Path(source).expanduser()
    if local.is_dir():
        return local.resolve()
    target = WORK_ROOT / name
    if not (target / ".git").is_dir():
        print("cloning " + str(source) + " ...")
        try:
            run_command(["git", "clone", "--depth", "1", source, target])
        except RuntimeError as exc:
            raise RuntimeError(
                str(exc) + "\n\n" + name + " could not be downloaded from " + str(source) + ". Put a "
                "repository this runtime can reach in the field at the top of this form, or the path of a "
                "folder you uploaded to this runtime (for example " + str(target) + ")."
            ) from None
    return target.resolve()


def package_dir(module):
    """The directory an importable package sits in, or None when it is not importable."""
    found = importlib.util.find_spec(module)
    if found is None or not found.origin:
        return None
    return Path(found.origin).resolve().parent


def colabsd_is_complete():
    """True when colabsd is importable *and* its config registry and bundled example came with it.

    Two layouts are both correct: an editable install leaves `config/` and `examples/` beside the
    package, a built wheel carries them inside it. Either answer counts; neither does.
    """
    package = package_dir("colabsd")
    if package is None:
        return False
    return any(
        (root / "config" / "best").is_dir() and (root / "examples").is_dir()
        for root in (package, package.parent)
    )


if not colabsd_is_complete():
    package_root = checkout(colabsd_repository, "ColabSeqDisplay")
    print("installing ColabSeqDisplay from " + str(package_root) + " ...")
    run_command([sys.executable, "-m", "pip", "install", "-q", "-e", package_root])
    importlib.invalidate_caches()
    if str(package_root) not in sys.path:
        sys.path.insert(0, str(package_root))

import colabsd
from colabsd.ui import core, main_workflow

runtime = core.detect_runtime()
WORK_DIR = WORK_ROOT / "colabsd_work"

if runtime.has_gpu:
    GPU_DESCRIPTION = str(runtime.gpu_name) + "  (" + format(runtime.gpu_memory_gb or 0.0, ".1f") + " GB)"
else:
    GPU_DESCRIPTION = "none — Runtime > Change runtime type > T4 GPU, then run this cell again"

print("colabsd " + colabsd.__version__ + "   from " + str(Path(colabsd.REPO_ROOT)))
print("GPU       " + GPU_DESCRIPTION)
print("files     " + str(WORK_DIR))
print("")

wizard = main_workflow.launch(work_dir=WORK_DIR)

In [ ]:
#@title **Click the run-button to unlock the test set and write the final report** { display-mode: "form" }

#@markdown ### Why this is a separate button
#@markdown - Everything the panel above reports is measured on the **validation** split, on purpose. You chose the backbone, the pooling and the number of runs by looking at those numbers, so they are no longer an honest estimate of how the model behaves on data nobody has looked at.
#@markdown - The **test** partition stays locked while all of that happens, and is read here: once, deliberately, by you. That is the whole point of holding it back — a number you consult while you are still making decisions stops being a test number.
#@markdown - **Every unlock is counted.** The count is written to `unlock.json` beside the run and printed in the report, so whoever reads the result can see how many times the test set was opened. Unlock once, at the end, when you have stopped changing things.
#@markdown - Nothing here retrains anything. If you unlock, then change something and train again, you unlock again — the count goes up, and the report says so.

try:
    from colabsd.ui import main_workflow, unlock
except ImportError:
    raise RuntimeError(
        "ColabSeqDisplay is not installed in this runtime yet. Run the cell above first: it installs the "
        "package and trains the model this cell reports on."
    ) from None

try:
    trained = wizard
except NameError:
    raise RuntimeError(
        "Run the cell above first, and train something in the panel it draws. This cell reports on that "
        "run: it reads the panel the cell above leaves behind as `wizard`, and without it "
        "there is nothing to unlock."
    ) from None

unlock.launch(
    trained,
    output_dir=main_workflow.run_dir(trained.state),
    work_dir=main_workflow.work_dir(trained.state),
)